# Notebook 4: Ultrasonic Sensing & Obstacle Avoidance

This is the last notebook in this introductory arc, and it's where
everything comes together: a **sensor** (input) drives **motors**
(output) automatically, with no human pressing a key or a button in the
loop. This is the first genuinely autonomous behavior in the project.

You'll build this up the same careful way every previous notebook has:

1. **The ultrasonic sensor, on its own** — reading distance, understanding
   what its readings mean (and don't mean) before it's ever wired to
   anything else.
2. **The simplest possible reaction** — `stop_if_obstacle()`: keep going,
   or stop, based on one threshold.
3. **One full reactive maneuver** — `avoid_obstacle()`: forward, and if
   something's in the way, stop, back up, turn, and resume.
4. **The autonomous loop** — `auto_drive()`: `avoid_obstacle()` called
   repeatedly, safely bounded so it can never run forever.

This notebook uses two already-tested project modules:
`src/hardware/ultrasonic.py` (sensor reading) and
`src/robot/obstacle_avoidance.py` (the behavior layer that composes the
sensor module together with Notebook 3's `motor.py` — notice this is the
project's first "behavior" module, one level above the hardware modules,
built by composing them rather than reimplementing anything). Both were
fully verified on the real Pi before this notebook was written, including a
live interrupt test confirming the robot genuinely stops if this notebook's
kernel is killed mid-drive.

This notebook drives real motors, same as Notebook 3, **and** introduces a
new physical hazard of its own (a sensor pin that outputs a higher voltage
than the Pi can safely accept) — so there are two separate safety sections
below, not one. Read both before wiring anything.

## Wiring the ultrasonic sensor (HC-SR04)

> ## SAFETY WARNING — ECHO IS A 5V SIGNAL, GPIO21 IS NOT
>
> The HC-SR04's **ECHO** pin outputs a **5V** signal. The Raspberry Pi's
> GPIO pins are **3.3V maximum** — connecting ECHO straight to GPIO21
> **will damage the Pi's input**.
>
> A **voltage divider** (or a proper logic-level shifter) between ECHO and
> GPIO21 is **mandatory**, every time, no exceptions. Never wire ECHO
> directly to the Pi.
>
> Remember from Notebook 1: *"never feed 5V into a GPIO input"* was
> explicitly planted there as foreshadowing for this exact moment — this is
> the sensor that warning was about.

### Connections

| HC-SR04 pin | Connect to | BCM pin | Physical pin |
|---|---|---|---|
| VCC | 5V | — | physical pin 2 (or 4) |
| TRIG | GPIO20 | 20 | physical pin 38 |
| ECHO | **through a voltage divider**, then GPIO21 | 21 | physical pin 40 |
| GND | GND | — | any GND pin, e.g. physical pin 39 (next to pin 40) |

A couple of details worth being explicit about:

- **VCC really is 5V, and that's fine.** The HC-SR04 module itself runs on
  5V power — that's normal and not the hazard here. The hazard is
  specifically ECHO's **output signal**, which swings up to 5V and is being
  fed *into* a GPIO **input** pin that can only safely read up to 3.3V.
  Power and signal are two different concerns; only the signal side needs
  the divider.
- **TRIG can be wired directly**, with no divider needed. TRIG is an input
  *to* the sensor (the Pi tells the sensor "take a reading now"), and the
  HC-SR04 accepts a 3.3V-level trigger pulse just fine — this project's
  `src/hardware/ultrasonic.py` docstring confirms this is the documented,
  intentional assumption for this specific sensor. It's only ECHO —
  signal flowing back *from* the sensor *into* the Pi — that's the 5V
  problem.
- **A simple two-resistor voltage divider** (for example, roughly 1kΩ and
  2kΩ in series, with GPIO21 connected at the midpoint) is enough to bring
  ECHO's 5V swing down to a safe ~3.3V. `PINOUT.md` in the project root
  documents this same divider approach if you want the reference numbers
  again later.

Double-check the divider is actually in the ECHO path — not TRIG, not
power — before applying power to the sensor for the first time.

## Two library defaults this project deliberately overrides

Before jumping into code, it's worth understanding two choices
`src/hardware/ultrasonic.py` makes, because they explain behavior you're
about to see and they're a good lesson in reading a library's actual
source instead of assuming defaults are fine — the same discipline
`motor.py` applied to `gpiozero.Motor`'s `enable` parameter in Notebook 3.

**`max_distance` — a hard clamp, not a soft cap.** `gpiozero.DistanceSensor`
computes `.distance` internally as `min(1.0, raw / max_distance) *
max_distance`. That means anything at or beyond `max_distance` doesn't read
back as "far away" — it reads back as *exactly* `max_distance`, every time,
indistinguishable from anything even farther. gpiozero's own default is
only 1 meter, too short for a room-scale robot. This project uses **3.0
meters** instead, chosen as a reasonable room-scale bound and documented in
the module rather than left as an unexplained number.

**`partial` — the difference between "returns immediately" and "can hang
forever."** With gpiozero's own default (`partial=False`), reading
`.distance` **blocks until real echo data has actually arrived** — verified
directly on this project's Pi to hang indefinitely with nothing wired, not
raise, not return `None`, not time out. For a sensor reading taken once,
that might just be annoying. For a sensor reading taken **inside a driving
loop**, it's a real safety hazard: a blocked read would freeze the whole
loop with the motors potentially still running and nothing left able to
call `stop()`. So this module always sets `partial=True`, which returns
immediately and falls back to **0.0 meters** ("something is very close")
whenever it doesn't yet have a real reading.

That fallback value — **0.0m, not some neutral "unknown" value** — is
itself a deliberate fail-safe choice: if the sensor is unwired, glitching,
or momentarily out of data, the robot's code sees "very close" rather than
"all clear." An obstacle-avoidance system that quietly assumed "clear" on
missing data would be far more dangerous than one that's overly cautious.
Keep this in mind for the whole notebook: **while the sensor is unwired,
every single reading will be 0.00m** — that's this fail-safe behavior
working exactly as designed, not a bug, and not something the code below
needs to special-case.

## Reading the sensor, on its own

### Explanation

Add `src/` to the path and import the ultrasonic module. `get_ultrasonic_sensor()`
creates a `gpiozero.DistanceSensor` on TRIG=GPIO20/ECHO=GPIO21 with the
`max_distance=3.0` / `partial=True` settings explained above.

In [ ]:
import sys
import time
sys.path.insert(0, '../src')

from hardware.ultrasonic import get_ultrasonic_sensor, read_distance, cleanup as cleanup_sensor

sensor = get_ultrasonic_sensor()
print("Ultrasonic sensor ready (TRIG=GPIO20, ECHO=GPIO21, max_distance=3.0m).")


### Expected result

`Ultrasonic sensor ready (TRIG=GPIO20, ECHO=GPIO21, max_distance=3.0m).`
printed, no error. (You may also see a background gpiozero warning about
"no echo received" if the sensor isn't wired yet — that's expected, not an
error, and matches exactly what the "two overridden defaults" section above
predicted.)

### Physical result

Nothing yet — creating the sensor object doesn't take a reading by itself.

### Explanation

Take a single reading with `read_distance(sensor)`, which just returns
`sensor.distance` — a float, in meters.

In [ ]:
print(f"Distance = {read_distance(sensor):.2f} m")


### Expected result

**If the sensor isn't wired yet**: `Distance = 0.00 m`, every time you run
this cell — this is the `partial=True` fail-safe default described above,
not a real measurement, and not a bug.

**Once the sensor is properly wired** (through its voltage divider): a real
distance in meters, roughly matching whatever is actually in front of the
sensor — for example `Distance = 1.20 m` in an open room, or a smaller
number if something is closer.

### Physical result

None — this only takes a reading, it doesn't move or change anything.

### Explanation

Loop a handful of readings, half a second apart, so you can watch the
values over time rather than just once.

In [ ]:
for i in range(8):
    distance_m = read_distance(sensor)
    print(f"Sample {i + 1}/8: Distance = {distance_m:.2f} m")
    time.sleep(0.5)


### Expected result

Eight `Sample N/8: Distance = X.XX m` lines, about 4 seconds total.

**Unwired**: every line reads `Distance = 0.00 m` — identical each time,
because there's no real echo ever coming back, just the same fail-safe
fallback each call.

**Wired**: the values should genuinely change between samples. Try this —
while this cell is running, wave your hand slowly toward and away from the
sensor and watch the printed numbers rise and fall in response. A steady,
unchanging non-zero number (e.g. always reading close to 3.00m) usually
means the sensor is aimed at open space beyond its `max_distance` clamp,
which is expected per the "hard clamp" behavior explained above, not an
error.

### Physical result

None from the robot itself — this section only reads the sensor, nothing
moves yet.

### Explanation

Release the sensor's pins for now — we'll create a fresh sensor object
again in the next section once motors are involved, the same "close, then
reopen fresh" pattern used in earlier notebooks.

In [ ]:
cleanup_sensor(sensor)
print("Sensor GPIO released.")


### Expected result

`Sensor GPIO released.` printed, no error.

### Physical result

None.

## Integrating with motors: `stop_if_obstacle()`

> ## SAFETY WARNING — MOTORS WILL MOVE
>
> **This notebook drives the robot's real motors**, same as Notebook 3.
> Before running any cell past this point:
>
> - **Lift the robot chassis off the ground/table**, or otherwise make sure
>   all four wheels can spin completely freely.
> - Keep it lifted/blocked for the **entire notebook** — the reactive loops
>   later in this notebook run the motors repeatedly and unattended for
>   several seconds at a time.
> - Know that `stop(motors)` (or interrupting the kernel) is always
>   available if anything looks wrong.

### Explanation

Create fresh `motors` and `sensor` objects, and import the simplest
building block from `src/robot/obstacle_avoidance.py`: `stop_if_obstacle()`.

Its logic is exactly one threshold check: **`distance > threshold` → do
nothing (path is clear); `distance < threshold` → stop the motors and
report `True`.** Nothing about backing up or turning yet — that's the next
section. The default threshold is **0.25m (25cm)**.

In [ ]:
from hardware.motor import get_motors, forward, stop
from robot.obstacle_avoidance import stop_if_obstacle

motors = get_motors()
sensor = get_ultrasonic_sensor()
print("Motors and sensor ready.")


### Expected result

`Motors and sensor ready.` printed, no error.

### Physical result

Nothing should move yet.

### Explanation

Start the robot driving forward, then repeatedly call `stop_if_obstacle()`
in a short loop and print what it reports each time.

**Because the sensor is unwired, every reading is 0.00m — closer than the
0.25m threshold — so expect `stop_if_obstacle()` to report `True` and stop
the motors on the very first check, almost immediately after `forward()`
starts them.** The robot will barely move before stopping itself. This is
the fail-safe default from the earlier section doing exactly what it's
designed to do: treat "no real reading yet" as "assume something is close
and stop." Once the sensor is genuinely wired and reading real distances,
this same cell would keep driving forward until something actually comes
within 25cm.

In [ ]:
forward(motors)

for i in range(20):
    obstacle_detected = stop_if_obstacle(motors, sensor)
    print(f"Check {i + 1}/20: obstacle detected = {obstacle_detected}")
    if obstacle_detected:
        break
    time.sleep(0.1)

stop(motors)  # safety net in case the loop finished without ever detecting one
print("Done.")


### Expected result

**Unwired**: `Check 1/20: obstacle detected = True` printed once, then
`Done.` — the loop breaks immediately after the first check.

**Wired, with a clear path**: several `obstacle detected = False` lines (up
to 20, about 2 seconds), then `Done.` via the safety-net `stop(motors)`
call at the end, since the loop ran its course without ever seeing an
obstacle.

**Wired, with something within 25cm**: `False` lines until something comes
into range, then one `True` line, then `Done.`.

### Physical result

**Unwired**: the wheels may twitch forward very briefly, then stop almost
immediately.

**Wired**: the robot should drive forward and keep going until either 20
checks pass (about 2 seconds) or something comes within about 25cm of the
sensor, at which point it stops.

## The full reactive maneuver: `avoid_obstacle()`

`stop_if_obstacle()` only ever stops — it has no way to get going again on
its own. `avoid_obstacle(motors, sensor)` builds one complete reactive cycle
on top of it:

```
                 ┌────────────────────────────────────────┐
                 ▼                                         │
        ┌────────────────┐        obstacle?        ┌───────────────┐
        │   drive FORWARD │ ── check distance ──►   │ path is clear │
        └────────────────┘                          └───────┬───────┘
                 ▲                                           │
                 │                                     (nothing else
     ┌───────────┴───────────┐                          happens - already
     │  TURN right (~0.5s)   │                           driving forward)
     └───────────▲───────────┘
                 │
     ┌───────────┴───────────┐
     │ back up (~0.5s), STOP │  ◄── obstacle detected (distance < threshold)
     └────────────────────────┘
```

In other words: check the sensor; if the path is clear, just keep/start
driving forward; if something's too close, **stop, back up briefly, turn
right briefly, stop**, and then resume driving forward regardless — so the
function always ends with the robot moving forward, whether or not it just
avoided something. That's deliberate: it means `auto_drive()` (next
section) can simply call `avoid_obstacle()` in a loop with no extra logic
of its own.

The turn direction (always right) and the backup/turn durations (~0.5s
each) are simple, fixed, predictable choices for teaching — not the only
reasonable design, and exactly the kind of thing this notebook's exercises
will ask you to change.

### Explanation

Import `avoid_obstacle` and call it once. `motors` and `sensor` are reused
from the previous section.

In [ ]:
from robot.obstacle_avoidance import avoid_obstacle

obstacle_handled = avoid_obstacle(motors, sensor)
print(f"Obstacle handled this call: {obstacle_handled}")
time.sleep(1)
stop(motors)


### Expected result

`Obstacle handled this call: True` (unwired — the 0.00m fail-safe reading
is always "too close") or `False` (wired, path genuinely clear), followed
by a 1-second pause and no further output before the cell ends.

### Physical result

**Unwired**: the full back-up → turn-right → forward sequence should play
out, briefly, since the sensor always reports "too close." Expect to see
the robot (if lifted) reverse briefly, pivot right briefly, then drive
forward — followed by the explicit `stop(motors)` in this cell.

**Wired, path clear**: the robot just drives forward (no back-up/turn),
then stops after the 1-second pause.

**Wired, obstacle present**: the same back-up → turn-right → forward
sequence as the unwired case, but genuinely triggered by something being
within 25cm.

## The autonomous loop: `auto_drive()`

`auto_drive(motors, sensor, ...)` just calls `avoid_obstacle()` repeatedly,
pausing briefly between checks — but two safety properties make it safe to
actually run, rather than a bare `while True` that could get away from you:

- **`max_duration_s` is required and always enforced.** There is no way to
  call `auto_drive()` with an unbounded run time — passing `None` or a
  non-positive value raises an error immediately rather than silently
  running forever. An autonomous loop that can only be stopped by you
  reacting in time (unplugging power, hitting Ctrl-C fast enough) is a real
  hazard; a loop that guarantees its own end time is not.
- **An optional `stop_condition` callable** is checked every iteration, so
  you can hand `auto_drive()` your own early-exit condition — a button's
  `is_pressed`, an iteration counter, anything callable with no arguments
  that returns `True` when it's time to stop.
- **Motors are stopped in a `finally` block.** If anything goes wrong mid-loop
  — an exception, or you interrupting the kernel — `motor_stop()` still
  runs before the function actually exits. QA verified this for real: a
  live `KeyboardInterrupt`/`SIGINT` sent to a running `auto_drive()` process
  was confirmed, via live GPIO polling on the still-running process (not
  just checked after the fact), to genuinely stop the motors.

**Because the sensor is unwired**, every check inside this loop will see
"obstacle" (the 0.00m fail-safe reading) and trigger the full back-up/turn
maneuver — so an unwired `auto_drive()` call won't calmly idle forward for
its whole duration; it will continuously back up, turn, and briefly move
forward, over and over, non-stop, for the entire time it runs. That's still
safe with the robot lifted, just noisier and busier than you might expect
from the name "auto_drive" — and it's exactly the same fail-safe behavior
you've already seen in the two sections above, just repeating continuously
instead of once.

### Explanation

Run `auto_drive()` for a short, deliberately brief demo — **5 seconds**,
well under the module's own 30-second default — so a first run stays quick
and easy to watch/interrupt. `max_duration_s` is passed explicitly here
rather than relying on the default.

In [ ]:
from robot.obstacle_avoidance import auto_drive

auto_drive(motors, sensor, max_duration_s=5)


### Expected result

A single line like `auto_drive: stopping (max_duration_s=5s reached).`
printed after about 5 seconds, then the cell finishes. No error.

### Physical result

**Unwired**: continuous back-up → turn-right → brief-forward cycling for
the full 5 seconds, then a final stop. **Wired**: the robot drives forward,
reactively avoiding anything that comes within 25cm, for 5 seconds, then
stops — try physically walking an object toward the sensor while this runs
to see it react.

Either way, the robot should be fully stopped by the time this cell's
output appears — that's the `finally`-block guarantee described above,
not just something that happens to work out.

### Explanation

Cleanup: stop everything and release all the GPIO pins — both the motor
module's and the sensor module's.

In [ ]:
from hardware.motor import cleanup as cleanup_motors

stop(motors)
cleanup_motors(motors)
cleanup_sensor(sensor)
print("Motors and sensor stopped and released.")


### Expected result

`Motors and sensor stopped and released.` printed, no error.

### Physical result

Everything should already be stopped from the previous cell's `finally`
block — this is a final, explicit safety net and pin release, same pattern
as every previous notebook's cleanup section.

## Recap

- `src/hardware/ultrasonic.py` overrides two `gpiozero.DistanceSensor`
  defaults for good, documented reasons: `max_distance=3.0` (room-scale,
  and a reminder that this is a hard clamp, not a soft cap) and
  `partial=True` (returns immediately, falling back to `0.0` — "very
  close" — instead of the alternative of blocking forever with nothing
  wired, which would be a real hazard inside a driving loop).
- **While unwired, every reading is `0.00m`** — the deliberate fail-safe
  default, not a bug, and something you saw affect every section of this
  notebook (motors stopping almost immediately, `avoid_obstacle()` always
  reporting an obstacle, `auto_drive()` continuously maneuvering).
- **ECHO is a 5V signal and must go through a voltage divider before
  GPIO21** — the concrete case the "never feed 5V into a GPIO input"
  warning from Notebook 1 was foreshadowing. TRIG, by contrast, is
  3.3V-safe and wired directly.
- `src/robot/obstacle_avoidance.py` is this project's first **behavior**
  module — built by composing the already-tested `motor.py` and
  `ultrasonic.py` hardware modules, not by reimplementing anything:
  `stop_if_obstacle()` (one threshold check) → `avoid_obstacle()` (one full
  forward/stop/back-up/turn/forward cycle) → `auto_drive()`
  (`avoid_obstacle()` looped, always time-bounded, always stopped in a
  `finally` block).
- An autonomous loop must never be able to run forever on its own —
  `auto_drive()`'s mandatory `max_duration_s`, optional `stop_condition`,
  and guaranteed `finally`-block stop are the concrete mechanisms this
  project uses to guarantee that, and QA verified the interrupt-safety part
  for real, on a live running process.

## Exercises

> ## SAFETY WARNING — MOTORS WILL MOVE
>
> **This notebook drives the robot's real motors**, same as Notebook 3.
> Before running any cell past this point:
>
> - **Lift the robot chassis off the ground/table**, or otherwise make sure
>   all four wheels can spin completely freely.
> - Keep it lifted/blocked for the **entire notebook** — the reactive loops
>   later in this notebook run the motors repeatedly and unattended for
>   several seconds at a time.
> - Know that `stop(motors)` (or interrupting the kernel) is always
>   available if anything looks wrong.

**1. Change the obstacle distance threshold.**
Call `avoid_obstacle(motors, sensor, threshold_m=0.40)` (more cautious —
reacts farther away) and, separately, a smaller value like `0.15` (more
aggressive — lets the robot get closer before reacting). Wrap either in the
same short loop/`auto_drive()` pattern from above and compare the behavior.
What tradeoffs do you notice between a larger and a smaller threshold?

**2. Improve the turning behavior.**
`avoid_obstacle()` always turns right by a fixed ~0.5s — the module's own
docstring calls "alternating or randomizing the turn direction" a natural
next exercise. Since there's no parameter for this, write your **own**
small function in this notebook (not editing `obstacle_avoidance.py`) that
copies `avoid_obstacle()`'s structure — `stop_if_obstacle()`, then back up,
then turn, then forward — but alternates between `left(motors, ...)` and
`right(motors, ...)` each time it's called (a simple toggle variable works:
flip a boolean each time an obstacle is handled). This is the same
"notebooks compose tested modules, new behaviors get written here" pattern
this project has followed throughout.

**3. Tune the maneuver durations.**
Try calling `avoid_obstacle(motors, sensor, backup_seconds=1.0,
turn_seconds=1.0)` for a longer, more dramatic avoidance maneuver, and
`backup_seconds=0.2, turn_seconds=0.2` for a quicker, subtler one. Once
wired, which feels more reliable for actually clearing an obstacle in your
space?

**4. (Optional, extra challenge) Start/stop `auto_drive()` with the button
from Notebook 2.**
`auto_drive()` accepts an optional `stop_condition` — any no-argument
callable that returns `True` when it's time to stop. Import `get_button()`
from `hardware.button` (Notebook 2), wire the button as you did there, and
pass `stop_condition=lambda: my_button.is_pressed` so a physical button
press ends the autonomous loop early, before `max_duration_s` is reached.
This combines three notebooks' worth of modules (button, motor, ultrasonic)
into one behavior, entirely at the notebook level.